In [1]:
import os
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
import math
import datetime as dt

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-07-01 16:19:52.726962


#### Constants

In [3]:
# name of step function
str_name = 'gen-xii-payload-parsing-jq'
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
int_n_requests_per_lambda = 100

Project: 20240423-gen-xii-payload-parsing
Task: 03_step_function


#### Make `rows` column in `df_requests.gzip`

In [4]:
%%time

# import data
str_filename = 'df_requests.gzip'
# str_uri = f's3://20240327-genxii-v2/01_feature_collection/01_get_requests_from_db/{str_filename}'
# demo uri
str_uri = 's3://20240423-gen-xii-payload-parsing/01_pull_local_db/df_requests.gzip'
df = pd.read_parquet(str_uri)

# show
df

CPU times: user 16.3 s, sys: 8.42 s, total: 24.7 s
Wall time: 43.5 s


,bigAccountId,dtmFunded,strRequest
10947,6507952,2023-01-03,"{""request_id"":""650795220692"",""rows"":[{""row_id""..."
10958,6490808,2023-01-03,"{""request_id"":""6490808132569"",""rows"":[{""row_id..."
10989,6511246,2023-01-03,"{""request_id"":""6511246346075"",""rows"":[{""row_id..."
11022,6500109,2023-01-03,"{""request_id"":""6500109894912"",""rows"":[{""row_id..."
11039,6499664,2023-01-03,"{""request_id"":""6499664617135"",""rows"":[{""row_id..."
...,...,...,...
35404,7373949,2023-12-01,"{""request_id"":""7373949863162"",""rows"":[{""row_id..."
35405,7307464,2023-11-28,"{""request_id"":""7307464105935"",""rows"":[{""row_id..."
35406,7327036,2023-12-11,"{""request_id"":""7327036323362"",""rows"":[{""row_id..."
35407,7335258,2023-12-07,"{""request_id"":""7335258911553"",""rows"":[{""row_id..."


In [5]:
# keep only the most recent payload
df.drop_duplicates(
    subset=['bigAccountId'],
    keep='last',
    inplace=True,
)
# show
df

,bigAccountId,dtmFunded,strRequest
10947,6507952,2023-01-03,"{""request_id"":""650795220692"",""rows"":[{""row_id""..."
10958,6490808,2023-01-03,"{""request_id"":""6490808132569"",""rows"":[{""row_id..."
10989,6511246,2023-01-03,"{""request_id"":""6511246346075"",""rows"":[{""row_id..."
11022,6500109,2023-01-03,"{""request_id"":""6500109894912"",""rows"":[{""row_id..."
11039,6499664,2023-01-03,"{""request_id"":""6499664617135"",""rows"":[{""row_id..."
...,...,...,...
35404,7373949,2023-12-01,"{""request_id"":""7373949863162"",""rows"":[{""row_id..."
35405,7307464,2023-11-28,"{""request_id"":""7307464105935"",""rows"":[{""row_id..."
35406,7327036,2023-12-11,"{""request_id"":""7327036323362"",""rows"":[{""row_id..."
35407,7335258,2023-12-07,"{""request_id"":""7335258911553"",""rows"":[{""row_id..."


In [6]:
# sort
df.sort_values(by='dtmFunded', ascending=True, inplace=True)

# show
df

,bigAccountId,dtmFunded,strRequest
10947,6507952,2023-01-03,"{""request_id"":""650795220692"",""rows"":[{""row_id""..."
11523,6504520,2023-01-03,"{""request_id"":""6504520665899"",""rows"":[{""row_id..."
11520,6519821,2023-01-03,"{""request_id"":""6519821836516"",""rows"":[{""row_id..."
11512,6514755,2023-01-03,"{""request_id"":""6514755774964"",""rows"":[{""row_id..."
11508,6512501,2023-01-03,"{""request_id"":""6512501796271"",""rows"":[{""row_id..."
...,...,...,...
34395,7306479,2024-02-09,"{""request_id"":""7306479330180"",""rows"":[{""row_id..."
34064,7359135,2024-02-09,"{""request_id"":""7359135706073"",""rows"":[{""row_id..."
35148,7351735,2024-02-28,"{""request_id"":""7351735580851"",""rows"":[{""row_id..."
30140,7272862,2024-03-04,"{""request_id"":""7272862920995"",""rows"":[{""row_id..."


In [7]:
# get min and max dates
dtm_min = df['dtmFunded'].min()
dtm_max = df['dtmFunded'].max()
print(f'Min date: {dtm_min.date()}; Max date: {dtm_max.date()}')

Min date: 2023-01-03; Max date: 2024-04-19


In [8]:
# get nrows
int_nrows = df.shape[0]

# divide by int_n_requests_per_lambda
int_n_lambdas = math.ceil(int_nrows / int_n_requests_per_lambda)
print(f'There will be {int_n_lambdas} lambdas')

There will be 230 lambdas


In [9]:
# create list to assign as new column
list_rows = list(np.tile(np.arange(1, int_n_lambdas+1), int_n_requests_per_lambda))
print(f'Length: {len(list_rows)}')
# make sure its the same length as df
list_rows = list_rows[:int_nrows]
print(f'Length: {len(list_rows)}')
# assign
df['rows'] = list_rows
# show
df

Length: 23000
Length: 22986


,bigAccountId,dtmFunded,strRequest,rows
10947,6507952,2023-01-03,"{""request_id"":""650795220692"",""rows"":[{""row_id""...",1
11523,6504520,2023-01-03,"{""request_id"":""6504520665899"",""rows"":[{""row_id...",2
11520,6519821,2023-01-03,"{""request_id"":""6519821836516"",""rows"":[{""row_id...",3
11512,6514755,2023-01-03,"{""request_id"":""6514755774964"",""rows"":[{""row_id...",4
11508,6512501,2023-01-03,"{""request_id"":""6512501796271"",""rows"":[{""row_id...",5
...,...,...,...,...
34395,7306479,2024-02-09,"{""request_id"":""7306479330180"",""rows"":[{""row_id...",212
34064,7359135,2024-02-09,"{""request_id"":""7359135706073"",""rows"":[{""row_id...",213
35148,7351735,2024-02-28,"{""request_id"":""7351735580851"",""rows"":[{""row_id...",214
30140,7272862,2024-03-04,"{""request_id"":""7272862920995"",""rows"":[{""row_id...",215


#### Make `df_idx.csv`

In [10]:
list_rows = list(df['rows'].value_counts().index)
df_idx = pd.DataFrame({'row': list_rows})
df_idx.sort_values(by='row', ascending=True, inplace=True)

# save
str_filename = 'df_idx.csv'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df_idx.to_csv(str_uri, index=False)

# show
df_idx

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:275: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,row
0,1
34,2
139,3
140,4
141,5
...,...
220,226
219,227
218,228
217,229


#### Subset and save

In [11]:
str_project, str_task

('20240423-gen-xii-payload-parsing', '03_step_function')

In [12]:
%%time

for int_row in tqdm(df_idx['row']):
    # subset
    df_tmp = df[df['rows'] == int_row].copy()
    # save
    str_filename = f'df_rows_{int_row}.gzip'
    str_uri = f's3://{str_project}/{str_task}/rows/{str_filename}'
    df_tmp.to_parquet(str_uri, compression='gzip')

100%|██████████| 230/230 [02:45<00:00,  1.39it/s]

CPU times: user 1min 32s, sys: 13.3 s, total: 1min 45s
Wall time: 2min 45s


#### Write `definition.json`

In [13]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "Map",
  "States": {
    "Map": {
      "Type": "Map",
      "ItemProcessor": {
        "ProcessorConfig": {
          "Mode": "DISTRIBUTED",
          "ExecutionType": "STANDARD"
        },
        "StartAt": "ParsePayloads",
        "States": {
          "ParsePayloads": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "OutputPath": "$.Payload",
            "Parameters": {
              "Payload.$": "$",
              "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:gen-xii-payload-parsing-jq:$LATEST"
            },
            "Retry": [
              {
                "ErrorEquals": [
                  "Lambda.ServiceException",
                  "Lambda.AWSLambdaException",
                  "Lambda.SdkClientException",
                  "Lambda.TooManyRequestsException"
                ],
                "IntervalSeconds": 1,
                "MaxAttempts": 3,
                "BackoffRate": 2
              }
            ],
            "End": true
          }
        }
      },
      "End": true,
      "Label": "Map",
      "MaxConcurrency": 1000,
      "ItemReader": {
        "Resource": "arn:aws:states:::s3:getObject",
        "ReaderConfig": {
          "InputType": "CSV",
          "CSVHeaderLocation": "FIRST_ROW"
        },
        "Parameters": {
          "Bucket": "20240423-gen-xii-payload-parsing",
          "Key": "03_step_function/df_idx.csv"
        }
      }
    }
  }
}

Writing definition.json


#### Make string definition

In [14]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

#### Create state machine

In [15]:
cls_client_sfn = boto3.client('stepfunctions')

In [16]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [17]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

{'DemoStepFunction': 'arn:aws:states:us-west-2:836690756591:stateMachine:DemoStepFunction',
 'MyStateMachine-fldl4s6of': 'arn:aws:states:us-west-2:836690756591:stateMachine:MyStateMachine-fldl4s6of',
 'christian-step-function': 'arn:aws:states:us-west-2:836690756591:stateMachine:christian-step-function',
 'gen-xi-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xi-retro-scoring',
 'gen-xii-payload-parsing-jq': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-payload-parsing-jq',
 'gen-xii-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring',
 'genxi-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxi-payload-parsing',
 'genxii-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxii-payload-parsing',
 'poc-step-genxii-lgd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3',
 'poc-step-genxii-pd-lambda-boto3': 'arn:aws:state

In [18]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

State machine gen-xii-payload-parsing-jq exists, it will be deleted
Deleting arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-payload-parsing-jq

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Mon, 01 Jul 2024 16:23:23 GMT',
                                      'x-amzn-requestid': '02307ee0-1b85-47a5-b90f-0628018cdf54'},
                      'HTTPStatusCode': 200,
                      'RequestId': '02307ee0-1b85-47a5-b90f-0628018cdf54',
                      'RetryAttempts': 0}}


In [19]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '130',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Mon, 01 Jul 2024 16:24:25 GMT',
                                      'x-amzn-requestid': 'fe8a70f7-a045-4b79-81ab-a1e6a9d0b3e1'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'fe8a70f7-a045-4b79-81ab-a1e6a9d0b3e1',
                      'RetryAttempts': 4},
 'creationDate': datetime.datetime(2024, 7, 1, 16, 24, 25, 430000, tzinfo=tzlocal()),
 'stateMachineArn': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-payload-parsing-jq'}


#### Describe state machine

In [20]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

State Machine ARN: arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-payload-parsing-jq
{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1602',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Mon, 01 Jul 2024 16:24:25 GMT',
                                      'x-amzn-requestid': 'a501160f-586c-4bea-ae6c-245401a41992'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'a501160f-586c-4bea-ae6c-245401a41992',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 7, 1, 16, 24, 25, 430000, tzinfo=tzlocal()),
 'definition': '{"Comment": "A description of my state machine", "StartAt": '
               '"Map", "States": {"Map": {"Type": "Map", "ItemProcessor": '
               '{"ProcessorConfig": {"Mode": "DISTRIBUTED", "ExecutionType": '
               '"STANDARD"}, 

#### Execute step function workflow

In [23]:
# start execution
dict_response = cls_client_sfn.start_execution(
    stateMachineArn=str_state_machine_arn,
)

#### Clean up 

In [22]:
os.remove('./definition.json')